In [9]:
import pandas as pd
import duckdb
import os
import os, json
from uuid import uuid4
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2 import service_account # based on google-auth library

In [10]:
try:
    file_data = json.load(open(os.path.expanduser("~/ServiceAccountsKey.json")))
    # (2) transform the content into crendentials object
    credentials = service_account.Credentials.from_service_account_info(file_data)
# (3) specify your usage of the credentials
    scoped_credentials = credentials.with_scopes(['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive'])
# (4) use the constrained credentials for authentication of gspread package
    gc = gspread.Client(auth=scoped_credentials)
    grela_gs = gc.open_by_url("https://docs.google.com/spreadsheets/d/1koPyP_XBITEnQs8hRcF4Si0DlPqGDeBkctpBhrtdkRk/edit?usp=sharing")
except:
    pass

In [6]:
conn = duckdb.connect("/srv/data/grela/grela_v0-3.duckdb", read_only=True)

In [8]:
# extract a subset of sentences with work level metadata, select on specific grela_id pattern
query = """
        SELECT s.*,
               w.*
        FROM sentences s
                 JOIN works w ON s.grela_id = w.grela_id
        WHERE w.textsource LIKE 'exprecce' \
        """

exprecce_sentences = conn.execute(query).fetchdf()

In [11]:
set_with_dataframe(grela_gs.add_worksheet("exprecce_sentences", 1,1), exprecce_sentences)